# Carga y Limpieza de Datos

In [7]:
import pandas as pd

# 1. Cargar el dataset usando la ruta relativa
ruta = '../rideshare_kaggle.csv'
df = pd.read_csv(ruta)

# 2. Filtrar los nulos de 'price' (categoría Taxi)
df_clean = df.dropna(subset=['price']).copy()

print(f"Filas originales: {len(df)}")
print(f"Filas limpias (sin Taxi): {len(df_clean)}")
print(f"Dimensiones del dataset: {df_clean.shape}")

Filas originales: 693071
Filas limpias (sin Taxi): 637976
Dimensiones del dataset: (637976, 57)


# División de Datos

In [8]:
from sklearn.model_selection import train_test_split

# Separar características (X) y variable objetivo (y)
X = df_clean.drop(columns=['price'])
y = df_clean['price']

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Seleccionar columnas numéricas para las pruebas iniciales
numeric_cols = X_train.select_dtypes(include=['number']).columns
X_train_num = X_train[numeric_cols].fillna(0)
X_test_num = X_test[numeric_cols].fillna(0)

print(f"Conjunto de Entrenamiento (X_train): {X_train_num.shape}")
print(f"Conjunto de Prueba (X_test):         {X_test_num.shape}")

Conjunto de Entrenamiento (X_train): (510380, 45)
Conjunto de Prueba (X_test):         (127596, 45)


# Modelo 1 - Baseline (Regresión Lineal)

In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Entrenar el modelo lineal simple
baseline_model = LinearRegression()
baseline_model.fit(X_train_num, y_train)

# Predicción y métricas
y_pred_base = baseline_model.predict(X_test_num)

rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)
r2_base = r2_score(y_test, y_pred_base)

print("--- MÉTRICAS MODELO BASELINE (Linear Regression) ---")
print(f"RMSE: ${rmse_base:.2f}")
print(f"MAE:  ${mae_base:.2f}")
print(f"R²:   {r2_base:.4f}")

--- MÉTRICAS MODELO BASELINE (Linear Regression) ---
RMSE: $8.48
MAE:  $6.95
R²:   0.1749


# Modelo 2 - Candidato Avanzado (Random Forest con limite de profundidad)

Nota de experimentación: Se entrenó un RandomForestRegressor con las variables numéricas. Debido a la alta carga computacional (7.5 min) y a un $R^2$ de 0.1506, se optó por utilizar HistGradientBoostingRegressor como modelo candidato principal por su eficiencia y compatibilidad con datos masivos.

In [10]:
from sklearn.ensemble import RandomForestRegressor

# Limitamos max_depth=15 para evitar que consuma toda la RAM de la Mac y sea ultrarrápido
rf = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)

print("Entrenando Random Forest...")
rf.fit(X_train_num, y_train)

y_pred_rf = rf.predict(X_test_num)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("\n--- MÉTRICAS MODELO CANDIDATO (Random Forest) ---")
print(f"RMSE: ${rmse_rf:.2f}")
print(f"MAE:  ${mae_rf:.2f}")
print(f"R²:   {r2_rf:.4f}")

Entrenando Random Forest...

--- MÉTRICAS MODELO CANDIDATO (Random Forest) ---
RMSE: $8.61
MAE:  $7.04
R²:   0.1506


In [11]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modelo optimizado para grandes volúmenes de datos (muy rápido en Mac)
hgb_model = HistGradientBoostingRegressor(random_state=42, max_iter=100)

print("Entrenando HistGradientBoostingRegressor...")
hgb_model.fit(X_train_num, y_train)

# Predicciones
y_pred_hgb = hgb_model.predict(X_test_num)

# Métricas
rmse_hgb = np.sqrt(mean_squared_error(y_test, y_pred_hgb))
mae_hgb = mean_absolute_error(y_test, y_pred_hgb)
r2_hgb = r2_score(y_test, y_pred_hgb)

print("\n--- MÉTRICAS MODELO CANDIDATO (HistGradientBoosting) ---")
print(f"RMSE: ${rmse_hgb:.2f}")
print(f"MAE:  ${mae_hgb:.2f}")
print(f"R²:   {r2_hgb:.4f}")

Entrenando HistGradientBoostingRegressor...

--- MÉTRICAS MODELO CANDIDATO (HistGradientBoosting) ---
RMSE: $8.45
MAE:  $6.94
R²:   0.1817


# One-Hot Encoding

In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Seleccionar características numéricas + variables categóricas clave
cols_to_use = numeric_cols.tolist() + ['name', 'cab_type']

# Extraer subconjuntos de entrenamiento y prueba
X_train_cat = X_train[cols_to_use].copy()
X_test_cat = X_test[cols_to_use].copy()

# 2. Aplicar One-Hot Encoding a las columnas de texto
X_train_encoded = pd.get_dummies(X_train_cat, columns=['name', 'cab_type'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test_cat, columns=['name', 'cab_type'], drop_first=True)

# Asegurar que ambos conjuntos tengan exactamente las mismas columnas
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Rellenar nulos
X_train_encoded = X_train_encoded.fillna(0)
X_test_encoded = X_test_encoded.fillna(0)

# 3. Entrenar HistGradientBoosting con las categorías incluidas
hgb_final = HistGradientBoostingRegressor(random_state=42)

print("Entrenando HistGradientBoosting con tipo de servicio...")
hgb_final.fit(X_train_encoded, y_train)

# 4. Predicciones y métricas
y_pred_final = hgb_final.predict(X_test_encoded)

rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
mae_final = mean_absolute_error(y_test, y_pred_final)
r2_final = r2_score(y_test, y_pred_final)

print("\n--- MÉTRICAS HISTGRADIENTBOOSTING CON CATEGÓRICAS ---")
print(f"RMSE: ${rmse_final:.2f}")
print(f"MAE:  ${mae_final:.2f}")
print(f"R²:   {r2_final:.4f}")

Entrenando HistGradientBoosting con tipo de servicio...

--- MÉTRICAS HISTGRADIENTBOOSTING CON CATEGÓRICAS ---
RMSE: $1.76
MAE:  $1.18
R²:   0.9644


# Exportar el Modelo Final

In [13]:
import joblib
import os

# 1. Crear carpeta para guardar los artefactos si no existe
os.makedirs('../modelos', exist_ok=True)

# 2. Guardar el modelo entrenado y las columnas de entrada esperadas
artefactos = {
    'modelo': hgb_final,
    'columnas_features': X_train_encoded.columns.tolist()
}

joblib.dump(artefactos, '../modelos/modelo_hgb_v1.joblib')
print("¡Modelo exportado exitosamente en '../modelos/modelo_hgb_v1.joblib'!")

¡Modelo exportado exitosamente en '../modelos/modelo_hgb_v1.joblib'!


In [14]:
import joblib

# 1. Cargar el objeto guardado
artefactos_cargados = joblib.load('../modelos/modelo_hgb_v1.joblib')

# 2. Extraer el modelo y las columnas esperadas
modelo_recuperado = artefactos_cargados['modelo']
columnas_recuperadas = artefactos_cargados['columnas_features']

print("¡Modelo cargado exitosamente en memoria!")
print(f"Tipo de modelo: {type(modelo_recuperado)}")
print(f"Cantidad de variables de entrada esperadas: {len(columnas_recuperadas)}")

¡Modelo cargado exitosamente en memoria!
Tipo de modelo: <class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingRegressor'>
Cantidad de variables de entrada esperadas: 57


In [15]:
# Tomamos la primera fila de prueba como ejemplo de entrada
ejemplo = X_test_encoded.iloc[[0]]

# Realizamos la predicción con el modelo recargado desde el archivo .joblib
precio_predicho = modelo_recuperado.predict(ejemplo)[0]
precio_real = y_test.iloc[0]

print(f"Precio Real del viaje:     ${precio_real:.2f} USD")
print(f"Precio Predicho (Joblib):  ${precio_predicho:.2f} USD")

Precio Real del viaje:     $7.00 USD
Precio Predicho (Joblib):  $8.64 USD
